# Multi-Modal Tri-Stream AVNet & AdapterNet: Cloud GPU Training Pipeline

This notebook trains the **Tri-Stream AVNet (AVNet-Mag)** and **AdapterNet-9Axis** dead-reckoning models with high GPU utilization.

> ⚠️ **CRITICAL FOR KAGGLE COMMIT MODE ('Save & Run All')**:
> In the right-hand sidebar under **Notebook options**:
> 1. **Accelerator**: Select **GPU T4 x 2** (or **GPU P100**).
> 2. **Internet**: Toggle to **On** (required for `git clone` and `git lfs pull`).
>
> All models, plots, and profiles will be saved directly into `/kaggle/working/` and appear in the **Output** tab upon commit completion.

In [ ]:
# 1. Environment & GPU Acceleration Setup
import os, sys, socket

# Check internet connectivity
try:
    socket.create_connection(("github.com", 443), timeout=5)
    print("Internet connectivity verified: ON")
except OSError:
    print("ERROR: Kaggle Internet is OFF! Please enable Internet in Notebook Settings -> Internet: On before committing.")

!nvidia-smi
!git lfs install 2>/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs && git lfs install)
!pip install -q torch torchvision torchaudio numpy scipy pandas matplotlib tqdm

In [ ]:
# 2. Clone Repository & Initialize Submodules with Git LFS
if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')
    print("Working directory: /kaggle/working")

# Clone or pull repository
if not os.path.exists('avnet'):
    print("Cloning dsainvg001/avnet...")
    !git clone --recurse-submodules https://github.com/dsainvg001/avnet.git
    %cd avnet
else:
    %cd avnet
    print("Pulling latest repository updates...")
    !git pull origin main

# Pull all Git LFS dataset files
!git lfs install
!git submodule update --init --recursive
!git -C data lfs install
!git -C data lfs pull

sys.path.append('.')
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using compute device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# 3. Verify Dataset & Git LFS Integrity
import os, glob
data_dir = 'data'

# Fallback if submodule directory is empty
if not os.path.exists(data_dir) or len(glob.glob('data/**/*.csv', recursive=True)) == 0:
    print("Cloning IO-VNBD dataset directly into data/...")
    !git clone https://github.com/onyekpeu/IO-VNBD.git data
    !cd data && git lfs install && git lfs pull && cd ..

csv_files = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
print(f"Total CSV files detected: {len(csv_files)}")
if csv_files:
    sample_file = csv_files[0]
    size_kb = os.path.getsize(sample_file) / 1024.0
    print(f"Sample file: {sample_file} ({size_kb:.1f} KB)")
    if size_kb < 1.0:
        print("WARNING: Pointer stub detected. Executing git lfs pull...")
        !git -C data lfs pull
    else:
        print("Git LFS verified: CSV files contain full real sensor data.")

In [ ]:
# 4. Auto-Discover Paired Files (S-*.csv paired with V-*.csv)
from avnet.dataset import discover_paired_iovnbd_files

pairs = discover_paired_iovnbd_files(data_dir)
print(f"Discovered unique synchronized pairs: {len(pairs)}")
if pairs:
    print(f"Sample S-file (Phone 9-Axis): {pairs[0][0]}")
    print(f"Sample V-file (Vehicle CAN):  {pairs[0][1]}")

In [ ]:
# 5. Create Train / Validation / Test DataLoaders (Optimized for High GPU Utilization)
from avnet.dataset import create_dataloaders

# batch_size=256 and num_workers=4 fully saturates GPU compute and eliminates CPU bottleneck
train_loader, val_loader, test_loader = create_dataloaders(
    root_dir=data_dir,
    window_size=20,     # 2.0 seconds at 10 Hz
    step=2,             # 90% overlap for dense training signals
    batch_size=256,     # High-throughput GPU batch size
    train_ratio=0.8,
    val_ratio=0.1,
    num_workers=4,
    limit_files=None
)

batch = next(iter(train_loader))
print("Sample Batch Tensors:")
print("  Accelerometer Stream:", batch['acc'].shape)         # (B, 3, 20)
print("  Gyroscope Stream:    ", batch['gyro'].shape)        # (B, 3, 20)
print("  Magnetometer Stream: ", batch['mag'].shape)         # (B, 3, 20)
print("  Target Forward Speed:", batch['target_speed'].shape) # (B, 1)
print("  Target Delta Quat:   ", batch['target_delta_q'].shape) # (B, 3)

In [ ]:
# 6. Train Multi-Modal Tri-Stream AVNet (Balanced Loss & Standardized Inputs)
from avnet.models.avnet import TriStreamAVNet
from avnet.train import train_tristream_avnet

model = TriStreamAVNet(window_size=20, hidden_dim=64)

EPOCHS = 10
LEARNING_RATE = 5e-4
LAMBDA_ATT = 50.0  # Balanced so speed and attitude losses contribute equally to gradients

trained_avnet, history = train_tristream_avnet(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    lambda_att=LAMBDA_ATT,
    checkpoint_dir='checkpoints',
    device=device
)

In [ ]:
# 7. Plot Training and Validation Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Multi-Task Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='#2563eb', lw=2)
if history['val_loss']:
    axes[0].plot(history['val_loss'], label='Val Loss', color='#dc2626', lw=2)
axes[0].set_title('Multi-Task Loss (Huber + Geodesic)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Speed RMSE
if history['speed_rmse']:
    axes[1].plot(history['speed_rmse'], label='Speed RMSE (m/s)', color='#16a34a', lw=2)
    axes[1].set_title('Validation Speed RMSE (m/s)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('m/s')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# Attitude Error
if history['att_error_deg']:
    axes[2].plot(history['att_error_deg'], label='Attitude Error (deg)', color='#9333ea', lw=2)
    axes[2].set_title('Validation Attitude Error (°)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Degrees')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# 8. Train AdapterNet-9Axis (Stable Innovation-Matching Optimization)
from avnet.models.avnet import AdapterNet9Axis
from avnet.train_adapter import train_adapter_offline
from avnet.dataset import load_iovnbd_csv

print("Loading sequences for AdapterNet offline optimization...")
sample_sequences = [load_iovnbd_csv(s, v) for s, v in pairs[:5] if v is not None]

adapter_model = AdapterNet9Axis(in_channels=9)
trained_adapter = train_adapter_offline(
    adapter_model=adapter_model,
    avnet_model=trained_avnet,
    data_sequences=sample_sequences,
    epochs=5,
    checkpoint_dir='checkpoints',
    device=device
)

In [ ]:
# 9. Closed-Loop InEKF Trajectory Benchmark
from main import run_evaluation

test_eval_data = load_iovnbd_csv(pairs[0][0], pairs[0][1])
results = run_evaluation(
    avnet_model=trained_avnet,
    adapter_model=trained_adapter,
    eval_data=test_eval_data,
    device=device,
    output_dir='results',
    max_eval_steps=10000
)

# Plot Estimated vs Ground Truth Path
pred_p = results['pred_positions']
gt_p = test_eval_data['gt_enu'][:len(pred_p)]

plt.figure(figsize=(10, 8))
plt.plot(gt_p[:, 0], gt_p[:, 1], 'r--', label='Ground Truth (Vehicle CAN/GPS)', lw=2.5)
plt.plot(pred_p[:, 0], pred_p[:, 1], 'b-', label='DMDVDR InEKF Dead Reckoning', lw=2.0)
plt.scatter(gt_p[0, 0], gt_p[0, 1], c='green', s=100, zorder=5, label='Start Point')
plt.title(f"Trajectory Dead Reckoning: Etrel={results['rel_metrics']['E_trel_percent']:.2f}% | ATE={results['ate']:.2f}m")
plt.xlabel('East (meters)')
plt.ylabel('North (meters)')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.savefig('results/trajectory_benchmark.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# 10. Package Model & Save Directly into Kaggle Outputs (/kaggle/working/)
import pickle, shutil

model_export_pkl = 'checkpoints/avnet_tristream_model.pkl'
export_package = {
    'model_name': 'TriStreamAVNet-Mag',
    'window_size': 20,
    'hidden_dim': 64,
    'state_dict': trained_avnet.state_dict(),
    'adapter_state_dict': trained_adapter.state_dict(),
    'final_metrics': {
        'val_loss': history['val_loss'][-1] if history['val_loss'] else None,
        'speed_rmse': history['speed_rmse'][-1] if history['speed_rmse'] else None,
        'att_deg': history['att_error_deg'][-1] if history['att_error_deg'] else None
    }
}

with open(model_export_pkl, 'wb') as f:
    pickle.dump(export_package, f)

# Copy all key artifacts directly to /kaggle/working/ for instant Kaggle Output tab download
kaggle_output_dir = '/kaggle/working'
if os.path.exists(kaggle_output_dir):
    for src in [
        'checkpoints/avnet_tristream_model.pkl',
        'checkpoints/best_avnet_tristream.pth',
        'checkpoints/best_avnet_tristream.pkl',
        'checkpoints/best_adapter.pth',
        'checkpoints/best_adapter.pkl',
        'checkpoints/training_curves.png',
        'results/calibration_profile.json',
        'results/trajectory_benchmark.png'
    ]:
        if os.path.exists(src):
            dest = os.path.join(kaggle_output_dir, os.path.basename(src))
            shutil.copy2(src, dest)
            print(f"Exported to Kaggle Output: {dest} ({os.path.getsize(dest)/1024:.1f} KB)")

print("\nAll model artifacts successfully exported to Kaggle Output directory!")

In [ ]:
# 11. Download Links & Output Verification
from IPython.display import FileLink, display

print("Generated files available in /kaggle/working/ (or local dir):")
target_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
for fname in [
    'avnet_tristream_model.pkl',
    'best_avnet_tristream.pth',
    'calibration_profile.json',
    'training_curves.png',
    'trajectory_benchmark.png'
]:
    fpath = os.path.join(target_dir, fname)
    if os.path.exists(fpath):
        print(f"  - {fname} ({os.path.getsize(fpath) / 1024:.1f} KB)")
        display(FileLink(fpath))

# Colab fallback
try:
    from google.colab import files
    files.download(os.path.join(target_dir, 'avnet_tristream_model.pkl'))
except ImportError:
    pass